# Customer Segmentation Using K-Means Clustering

**Assignment Description:** Perform K-Means clustering on a mall dataset and describe customer groups.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Load or Generate Mall Dataset
Since we do not have an external CSV currently, we will generate a synthetic dataset similar to the popular 'Mall_Customers' dataset, featuring Annual Income and Spending Score.

In [ ]:
# Generating synthetic mall customer data (similar to standard Mall Customers dataset)
np.random.seed(42)

# Define 5 typical customer groups based on Income and Spending Score
centers = [[15, 39], [15, 81], [55, 50], [85, 15], [85, 85]] # [Income, Spending Score]
stds = [8, 8, 15, 10, 10]

X_simulated = []
for center, std in zip(centers, stds):
    X_simulated.append(np.random.normal(loc=center, scale=std, size=(40, 2)))

X = np.vstack(X_simulated)
X = np.clip(X, 1, 130) # Clip values to be positive and within realistic ranges

df = pd.DataFrame(X, columns=['Annual_Income_k$', 'Spending_Score'])

# Add some random IDs, Gender and Age for realism
df['CustomerID'] = range(1, len(df) + 1)
df['Gender'] = np.random.choice(['Male', 'Female'], len(df))
df['Age'] = np.random.randint(18, 70, len(df))

# Reorder columns
df = df[['CustomerID', 'Gender', 'Age', 'Annual_Income_k$', 'Spending_Score']]
df.head()

## 2. Exploratory Data Analysis (EDA)
Let's visualize the distribution of Annual Income vs. Spending Score.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual_Income_k$', y='Spending_Score', s=60, hue='Gender')
plt.title('Customer Data: Annual Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.show()

## 3. Finding Optimal Number of Clusters (Elbow Method)
We use the Within-Cluster Sum of Squares (WCSS) to find the 'elbow' point which indicates the optimal number of clusters (`k`).

In [ ]:
# We will cluster based on Annual Income and Spending Score
X_cluster = df[['Annual_Income_k$', 'Spending_Score']].values

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_cluster)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('The Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.xticks(range(1, 11))
plt.show()

From the Elbow method, we can observe that the elbow roughly occurs at `k=5`. We will proceed with 5 clusters.

## 4. Training K-Means Model

In [ ]:
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(X_cluster)


## 5. Visualizing the Clusters

In [ ]:
plt.figure(figsize=(10, 7))

colors = ['red', 'blue', 'green', 'cyan', 'magenta']
for i in range(5):
    plt.scatter(X_cluster[df['Cluster'] == i, 0], X_cluster[df['Cluster'] == i, 1], 
                s=60, c=colors[i], label=f'Cluster {i}')

# Plotting the centroids
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            s=200, c='yellow', marker='*', edgecolor='black', label='Centroids')

plt.title('Clusters of Customers')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.show()

## 6. Describe Customer Groups

Based on the clustering above, we can describe the 5 distinct customer groups:

1.  **Low Income, Low Spending:** Customers with low annual income who also spend less. These might be careful spenders saving their money.
2.  **Low Income, High Spending:** Customers with low annual income but a high spending score. These are potential impulse buyers.
3.  **Average Income, Average Spending:** Customers in the middle for both income and spending. This is usually the largest demographic.
4.  **High Income, Low Spending:** Customers with high income who have a low spending score. These customers are frugal or save their money.
5.  **High Income, High Spending:** Customers with high income who spend heavily. These are prime targets for premium marketing campaigns.